# Phase 4 — Model Training
**Goal:** Train the ResNet50 + custom head model, monitor for overfitting, and save the best weights.

**Rubric relevance:** Technical Depth — justified hyperparameter choices, training/validation loss curves as evidence of overfitting mitigation, model checkpointing.

In [ ]:
import sys, os, json
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight

# Add the project root to Python's path so we can import from src/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.preprocess import get_generators, TRAIN_DIR
from src.model import build_model

# Paths
BASE_DIR      = os.path.join(os.getcwd(), '..')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH    = os.path.join(MODELS_DIR, 'ambiance_model.h5')
HISTORY_PATH  = os.path.join(MODELS_DIR, 'training_history.json')
CURVES_PATH   = os.path.join(MODELS_DIR, 'training_curves.png')

## 1. Load data generators

In [ ]:
train_gen, val_gen, test_gen = get_generators()

NUM_CLASSES = train_gen.num_classes
assert NUM_CLASSES == 19, f"Expected 19 classes, got {NUM_CLASSES} — check for missing or misnamed folders."

print(f"Training images:   {train_gen.samples}")
print(f"Validation images: {val_gen.samples}")
print(f"Test images:       {test_gen.samples}")
print(f"Classes ({NUM_CLASSES}): {list(train_gen.class_indices.keys())}")

## 2. Class balance check
If one class has far more images than others the model will be biased toward predicting it.
We compute class weights to compensate — the model penalises mistakes on smaller classes more heavily.

In [ ]:
class_names = list(train_gen.class_indices.keys())

# Count images per class folder directly from disk (before the val split)
counts = [len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in class_names]

plt.figure(figsize=(14, 4))
plt.bar(class_names, counts, color='steelblue')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.ylabel('Images')
plt.title('Training images per class')
plt.tight_layout()
plt.show()

print(f"Min: {min(counts)}  |  Max: {max(counts)}  |  Ratio: {max(counts)/min(counts):.2f}x")

# compute_class_weight assigns higher weight to underrepresented classes.
# 'balanced' sets weight = total_samples / (num_classes * class_count), so rare classes get upweighted.
# This is passed to model.fit() so the loss function penalises mistakes on smaller classes more.
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_gen.classes
)
class_weight_dict = dict(enumerate(class_weights_array))

print("\nClass weights (higher = model penalises mistakes more):")
for idx, name in enumerate(class_names):
    print(f"  {idx:>2}  {name:<25}  weight: {class_weight_dict[idx]:.3f}")

## 3. Build the model

In [ ]:
# build_model() returns (model, base_model).
# We keep base_model as a separate reference — needed for fine-tuning in Phase 4b.
model, base_model = build_model(num_classes=NUM_CLASSES)
model.summary()

## 4a. Train the custom head (frozen ResNet50 base)
In this phase only the custom ANN head is trained — the ResNet50 base stays frozen.
We train until val_loss stops improving, then move to fine-tuning.

In [ ]:
# EarlyStopping: stops after 5 epochs with no val_loss improvement, then rewinds to the best weights.
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

# ModelCheckpoint: saves to disk only on a new best val_loss.
checkpoint = ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1)

# ReduceLROnPlateau: halves the learning rate after 3 stalled epochs to help escape flat loss regions.
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

history_head = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    class_weight=class_weight_dict,  # Compensates for any class imbalance
    callbacks=[early_stopping, checkpoint, reduce_lr]
)

print(f"Phase 4a complete. Best val accuracy: {max(history_head.history['val_accuracy'])*100:.1f}%")

## 4b. Fine-tune the top ResNet50 layers
Now that the custom head has learned to classify interior styles, we unfreeze the top 30 layers
of ResNet50 and continue training with a much lower learning rate.

**Why fine-tune?**
The frozen ResNet50 layers learned general image features (edges, textures) from ImageNet.
Fine-tuning lets those top layers also adapt to the specific visual patterns of interior design styles
(colour palettes, materials, structural elements) — pushing accuracy significantly higher.

**Why only the top 30 layers?**
Early ResNet layers detect universal low-level features (edges, gradients) that transfer perfectly.
Only the later layers need to adapt to our domain. Unfreezing all 175 layers with our dataset size would overfit.

**Why 1e-5 learning rate?**
10× lower than the head training rate. The ResNet weights are already good — we nudge them, not overwrite them.

In [ ]:
# Unfreeze the top 30 layers of ResNet50 (the deeper, more task-specific feature extractors).
# base_model is the ResNet50 object returned by build_model() — its layers are shared with our main model,
# so setting trainable=True here affects the weights inside model automatically.
for layer in base_model.layers[-30:]:
    layer.trainable = True

print(f"Trainable layers after unfreezing: {sum(1 for l in model.layers if l.trainable)}")

# Recompile with a much lower learning rate — must recompile after changing trainable flags
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Recreate callbacks — reset patience counters so fine-tuning gets a fresh run
early_stopping_ft = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
checkpoint_ft     = ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr_ft      = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

history_fine = model.fit(
    train_gen,
    epochs=20,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=[early_stopping_ft, checkpoint_ft, reduce_lr_ft]
)

print(f"Phase 4b complete. Best val accuracy: {max(history_fine.history['val_accuracy'])*100:.1f}%")

## 5. Plot training curves
Both phases are plotted together. The vertical line marks where fine-tuning began.

In [ ]:
# Combine both phases into one continuous history
acc      = history_head.history['accuracy']     + history_fine.history['accuracy']
val_acc  = history_head.history['val_accuracy'] + history_fine.history['val_accuracy']
loss     = history_head.history['loss']         + history_fine.history['loss']
val_loss = history_head.history['val_loss']     + history_fine.history['val_loss']

phase_boundary = len(history_head.history['loss'])  # Epoch where fine-tuning started

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss,     label='Training loss')
axes[0].plot(val_loss, label='Validation loss')
axes[0].axvline(x=phase_boundary, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()

axes[1].plot(acc,     label='Training accuracy')
axes[1].plot(val_acc, label='Validation accuracy')
axes[1].axvline(x=phase_boundary, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('Training vs Validation — Phase 4a (frozen) + 4b (fine-tuned)', fontsize=12)
plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=150)
plt.show()
print(f"Curves saved to: {CURVES_PATH}")

# Save history to JSON so curves can be re-plotted without retraining
full_history = {
    'accuracy': acc, 'val_accuracy': val_acc,
    'loss': loss,    'val_loss': val_loss,
    'phase_boundary': phase_boundary
}
with open(HISTORY_PATH, 'w') as f:
    json.dump(full_history, f)
print(f"History saved to: {HISTORY_PATH}")

# Final unbiased test set evaluation — never seen during training or validation
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print(f"\n── Results ────────────────────────────────")
print(f"Head training epochs:  {len(history_head.history['loss'])}")
print(f"Fine-tuning epochs:    {len(history_fine.history['loss'])}")
print(f"Best val accuracy:     {max(val_acc)*100:.1f}%")
print(f"Test accuracy:         {test_acc*100:.1f}%")
print(f"Model saved to:        {MODEL_PATH}")
print(f"\nNext step: 03_evaluation.ipynb — F1 per class and confusion matrix.")